# Practical Lab-8: Support Vector Machine (SVM) Classification

**Student Name:** Hanshal Bobate  
**Roll No.:** 40  
**Section:** B  
**Batch:** B3  
**Branch:** Computer Science Engineering — AI/ML  
**Aim:** To implement the Support Vector Machine (SVM) algorithm for classification.

**Objective:** To train and evaluate linear and RBF SVM classifiers, then tune model hyperparameters for machine-failure prediction using the AI4I 2020 Predictive Maintenance Dataset.

## 1. Import Libraries

Import Pandas and NumPy for data handling, Matplotlib and Seaborn for visualization, and scikit-learn tools for splitting data, preprocessing features, building SVM pipelines, tuning hyperparameters, and evaluating classification performance.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)
import warnings
warnings.filterwarnings('ignore')

## 2. Load the Dataset

Load the AI4I 2020 Predictive Maintenance dataset from `ai4i2020.csv`. The code checks common notebook locations and gives a clear message if the CSV has not been placed in an accessible location. Keep the CSV alongside this notebook when possible.

In [ ]:
from pathlib import Path

# Look for the supplied dataset in common notebook working directories.
candidate_paths = [
    Path("ai4i2020.csv"),
    Path("/content/ai4i2020.csv"),  # Google Colab working directory
    Path("/mnt/data/ai4i2020.csv")  # common uploaded-file directory
]
dataset_path = next((p for p in candidate_paths if p.is_file()), None)

if dataset_path is None:
    raise FileNotFoundError(
        "Could not find 'ai4i2020.csv'. Upload/copy the dataset into the same "
        "folder as this notebook, or update candidate_paths with its actual location."
    )

df = pd.read_csv(dataset_path)
print(f"Loaded dataset from: {dataset_path.resolve()}")
print(f"Dataset shape: {df.shape}")

## 3. Preview the First Records

Display the first five rows to verify the dataset loaded and inspect the feature names and example values.

In [ ]:
df.head()

## 4. Preview the Last Records

Display the last five rows as a quick check of the records at the end of the dataset.

In [ ]:
df.tail()

## 5. Dataset Dimensions

Check the number of rows and columns. Rows represent machine observations; columns represent identifiers, measurements, and target information.

In [ ]:
df.shape

## 6. Column Names

List the dataset columns so the feature names and target column can be identified precisely.

In [ ]:
df.columns

## 7. Data Types

Inspect each column's data type to distinguish numerical measurements from categorical or identifier columns.

In [ ]:
df.dtypes

## 8. DataFrame Summary

Use `info()` to inspect non-null counts, data types, and memory usage. This is useful for detecting missing values and unexpected types.

In [ ]:
df.info()

## 9. Descriptive Statistics

Summarize numerical columns using count, mean, standard deviation, quartiles, minimum, and maximum to understand typical values and spread.

In [ ]:
df.describe()

## 10. Missing-Value Check

Count missing values in each column. Missing data should be understood before fitting the classifier.

In [ ]:
df.isnull().sum()

## 11. Duplicate-Record Check

Count exact duplicate rows. This reports duplicates without automatically removing observations.

In [ ]:
df.duplicated().sum()

## 12. Product-Type Categories

List the unique values in `Type` to understand the categories that will later be one-hot encoded.

In [ ]:
df["Type"].unique()

## 13. Target-Class Distribution

Count each value of `Machine failure`. The target may be imbalanced, so later evaluation should consider precision, recall, F1-score, and the confusion matrix in addition to accuracy.

In [ ]:
df["Machine failure"].value_counts()

## 14. Univariate Analysis: Numerical Histograms

Plot histograms for the five numerical machine measurements. These help reveal spread, skewness, clusters, and potential unusual values.

In [ ]:
numerical_features = [
    "Air temperature [K]",
    "Process temperature [K]",
    "Rotational speed [rpm]",
    "Torque [Nm]",
    "Tool wear [min]"
]

df[numerical_features].hist(figsize=(15, 10), bins=30)
plt.show()

## 15. Target Distribution Plot

Plot the number of failure and non-failure records. This visualizes class imbalance in the prediction target.

In [ ]:
plt.figure(figsize=(6,4))
sns.countplot(x="Machine failure", data=df)

plt.title("Distribution of Machine Failure")
plt.xlabel("Machine Failure")
plt.ylabel("Number of Machines")
plt.show()

## 16. Select Numerical Features

Define the numerical measurement columns used in exploratory plots and preprocessing. These are sensor/operational measurements rather than identifiers.

In [ ]:
numerical_features = [
    "Air temperature [K]",
    "Process temperature [K]",
    "Rotational speed [rpm]",
    "Torque [Nm]",
    "Tool wear [min]"
]


## 17. Numerical Histograms (Repeated View)

Display the distributions of the selected numerical features again. This cell is retained from the original notebook.

In [ ]:
df[numerical_features].hist(figsize=(15, 10), bins=30)
plt.show()


## 18. Outlier Inspection

Use boxplots to visualize medians, interquartile ranges, and observations beyond the whiskers for each numerical feature. Potential outliers are not automatically removed because extreme readings may be meaningful for predictive maintenance.

In [ ]:
plt.figure(figsize=(12,8))

for i, column in enumerate(numerical_features,1):
    plt.subplot(2,3,i)
    sns.boxplot(y=df[column])
    plt.title(column)

plt.tight_layout()
plt.show()

## 19. Correlation Heatmap

Visualize pairwise linear correlations among the numerical features and the machine-failure target. Correlation indicates association, not causation, and may not capture nonlinear patterns.

In [ ]:
plt.figure(figsize=(12,7))
correlation = df[numerical_features + ["Machine failure"]].corr()
sns.heatmap(correlation, annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Heatmap")
plt.show()

## 20. Numerical Features vs. Machine Failure

Compare each numerical feature's distribution across failure and non-failure records using boxplots. These plots help identify possible differences between the target groups.

In [ ]:
for column in numerical_features:
  plt.figure(figsize=(7,4))
  sns.boxplot(x="Machine failure", y=column, data=df)
  plt.title(f"{column} Vs Machine failure")
  plt.show()

## 21. Machine Failure by Product Type

Compare failure and non-failure counts across product types. Raw counts can depend on group size, so they should not automatically be interpreted as failure rates.

In [ ]:
plt.figure(figsize=(7,5))
sns.countplot(x="Type",hue="Machine failure",data=df)
plt.title("Machine failure by product Type")
plt.xlabel("Product type")
plt.ylabel("Count")
plt.show()

## 22. Prepare Input Features (X)

Select the five numerical measurements and categorical `Type` column as the model inputs. Identifier columns and the target are excluded from `X`.

In [ ]:
X= df[
    [
        "Air temperature [K]",
        "Process temperature [K]",
        "Rotational speed [rpm]",
        "Torque [Nm]",
        "Tool wear [min]",
        "Type"
]
]

## 23. Prepare the Target (y)

Set `y` to the `Machine failure` column, which contains the class label the SVM will learn to predict.

In [ ]:
y=df["Machine failure"]

## 24. Verify Input and Target Shapes

Print the dimensions of `X` and `y` to confirm that the input features and target have matching observation counts.

In [ ]:
print("Input shape",X.shape)
print("Output shape",y.shape)

## 25. Identify Numerical and Categorical Columns

Separate feature names into numerical and categorical lists. This allows each group to receive an appropriate preprocessing transformation.

In [ ]:
categorical_features =["Type"]
numerical_features =[
    "Air temperature [K]",
    "Process temperature [K]",
    "Rotational speed [rpm]",
    "Torque [Nm]",
    "Tool wear [min]"
]

## 26. Split into Training and Testing Sets

Split the data into 80% training and 20% testing subsets. `stratify=y` preserves the target-class proportions in both subsets, and `random_state=42` makes the split reproducible.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])

## 27. Build the Preprocessing Transformer

Standardize numerical features with `StandardScaler` and convert the categorical `Type` feature into one-hot indicators. `handle_unknown='ignore'` prevents errors if an unseen category appears during transformation.

In [ ]:
processor = ColumnTransformer(
    transformers=[
        ("num",StandardScaler(),numerical_features),
        ("cat",OneHotEncoder(handle_unknown='ignore'),categorical_features)
    ]
)

## 28. Train the Linear SVM

Create a pipeline that applies preprocessing and then fits an SVM with a linear kernel. Keeping preprocessing inside the pipeline ensures the transformations are fitted using the training data only.

In [ ]:
linear_svm = Pipeline(
    steps=[("preprocessor", processor), ("classifier", SVC(kernel="linear"))]
)
linear_svm.fit(X_train, y_train)

## 29. Predict with the Linear SVM

Use the trained linear SVM pipeline to predict machine-failure labels for the held-out test set.

In [ ]:
y_pred_linear = linear_svm.predict(X_test)

## 30. Evaluate the Linear SVM

Calculate accuracy, precision, recall, and F1-score for the linear SVM. These metrics provide a more complete view than accuracy alone, especially when failures are rare.

In [ ]:
accuracy_linear = accuracy_score(y_test, y_pred_linear)
precision_linear = precision_score(y_test, y_pred_linear)
recall_linear = recall_score(y_test, y_pred_linear)
f1_linear = f1_score(y_test, y_pred_linear)

print("Accuracy:", accuracy_linear)
print("Precision:", precision_linear)
print("Recall:", recall_linear)
print("F1 Score:", f1_linear)

## 31. Linear SVM Classification Report

Display precision, recall, F1-score, and support for each target class. `zero_division=0` avoids undefined-metric warnings if a class receives no predictions.

In [ ]:
print(
    classification_report(
        y_test,
        y_pred_linear,
        target_names=["No Failure", "Failure"],
        zero_division=0
    )
)

## 32. Linear SVM Confusion Matrix

Plot the confusion matrix to show correct predictions and errors for both classes. In particular, false negatives represent failures that the model missed.

In [ ]:
cm_linear = confusion_matrix(y_test, y_pred_linear)
plt.figure(figsize=(6, 4))
sns.heatmap(cm_linear, annot=True, fmt="d", cmap="Set2",
            xticklabels=["No Failure","Failure"],
            yticklabels=["No Failure","Failure"]
            )
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix - Linear SVM")
plt.show()

## 33. Train the RBF SVM

Train a second pipeline using the Radial Basis Function (RBF) kernel, which can learn nonlinear decision boundaries. The same preprocessing strategy and training split are used for a fair comparison.

In [ ]:
rbf_svm = Pipeline(
    steps=[("preprocessor", processor), ("classifier", SVC(kernel="rbf"))]
)
rbf_svm.fit(X_train, y_train)

## 34. Predict with the RBF SVM

Generate predictions for the test set using the fitted RBF-kernel SVM.

In [ ]:
y_pred_rbf = rbf_svm.predict(X_test)

## 35. Evaluate the RBF SVM

Calculate accuracy, precision, recall, and F1-score for the RBF model so its results can be compared with the linear SVM.

In [ ]:
accuracy_rbf = accuracy_score(y_test, y_pred_rbf)
precision_rbf = precision_score(y_test, y_pred_rbf)
recall_rbf = recall_score(y_test, y_pred_rbf)
f1_rbf = f1_score(y_test, y_pred_rbf)

print("RBF SVM Results")
print("Accuracy:", accuracy_rbf)
print("Precision:", precision_rbf)
print("Recall:", recall_rbf)
print("F1 Score:", f1_rbf)

## 36. RBF SVM Classification Report

Show per-class precision, recall, F1-score, and support for the RBF SVM.

In [ ]:
print(classification_report(y_test,y_pred_rbf,target_names= ["No Failure","Failure"],zero_division=0))

## 37. RBF SVM Confusion Matrix

Visualize correct predictions and misclassifications for the RBF model. Compare the false-negative count with the linear model.

In [ ]:
cm_rbf = confusion_matrix(y_test, y_pred_rbf)
plt.figure(figsize=(6, 4))
sns.heatmap(cm_rbf, annot=True, fmt="d", cmap="Set2",
            xticklabels=["No Failure","Failure"],
            yticklabels=["No Failure","Failure"]
            )
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix - RBF SVM")
plt.show()

## 38. Compare Linear and RBF SVM Results

Create a table containing the evaluation metrics for both kernels. This makes it easier to compare performance using multiple measures.

In [ ]:
comparison =pd.DataFrame({
    "Model": ["Linear SVM", "RBF SVM"],
    "Accuracy": [accuracy_linear, accuracy_rbf],
    "Precision": [precision_linear, precision_rbf],
    "Recall": [recall_linear, recall_rbf],
    "F1 Score": [f1_linear, f1_rbf]
})
comparison

## 39. Visualize Kernel Comparison

Plot the metrics for the linear and RBF models on a common scale. Use the chart alongside the confusion matrices and per-class reports, not as a replacement for them.

In [ ]:
comparison_plot = comparison.set_index("Model")

comparison_plot.plot(
    kind="bar",
    figsize=(10,6)
)
plt.title("Comparison of SVM Kernels")
plt.xlabel("SVM Kernel")
plt.ylabel("Score")
plt.ylim(0,1)
plt.xticks(rotation=0)
plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.show()

## 40. Define the SVM Tuning Pipeline

Create a general pipeline that combines the same preprocessing transformer with an SVM classifier whose kernel and hyperparameters will be selected by cross-validation.

In [ ]:
from sklearn.pipeline import Pipeline
pipeline = Pipeline(
    steps=[
        ("preprocessor", processor),
        ("classifier", SVC())
    ]
)

## 41. Define the Hyperparameter Search Space

Specify candidate values for the linear and RBF kernels, including regularization parameter `C` and RBF `gamma`. Grid search will evaluate the combinations defined here.

In [ ]:
param_grid = [
    {"classifier__kernel": ["linear"], "classifier__C": [0.1, 1, 10]},
    {"classifier__kernel": ["rbf"], "classifier__C": [1, 10, 50], "classifier__gamma": ["scale", 0.05, 0.1]}
]


## 42. Configure GridSearchCV

Set up five-fold cross-validation and use F1-score to select hyperparameters. This emphasizes the balance between precision and recall for the failure class. `n_jobs=1` runs one job at a time.

In [ ]:
grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    scoring="f1",
    cv=5,
    n_jobs=1,
    verbose=1
)


## 43. Run Hyperparameter Search

Fit the grid search on the training data. Cross-validation selects the candidate configuration with the best mean F1-score.

In [ ]:
grid_search.fit(X_train, y_train)
print("GridSearchCV completed successfully.")

## 44. Inspect the Best Parameters

Display the hyperparameter combination selected by GridSearchCV.

In [ ]:
print("Best Parameters:")
print(grid_search.best_params_)

## 45. Inspect Best Cross-Validation F1

Display the best mean F1-score obtained across the cross-validation folds for the selected configuration.

In [ ]:
print("Best Cross-Validation F1 Score:",grid_search.best_score_)

## 46. Predict with the Tuned SVM

Retrieve the best fitted pipeline and use it to predict the held-out test set. The test set is kept separate from hyperparameter selection.

In [ ]:
best_model = grid_search.best_estimator_
y_pred_tuned = best_model.predict(X_test)

## 47. Evaluate the Tuned SVM

Calculate accuracy, precision, recall, and F1-score for the tuned model on the test set.

In [ ]:
accuracy_tuned = accuracy_score(y_test, y_pred_tuned)
precision_tuned = precision_score(y_test, y_pred_tuned, zero_division=0)
recall_tuned = recall_score(y_test, y_pred_tuned, zero_division=0)
f1_tuned = f1_score(y_test, y_pred_tuned, zero_division=0)

print("Tuned Model Results")
print("Accuracy:", accuracy_tuned)
print("Precision:", precision_tuned)
print("Recall:", recall_tuned)
print("F1 Score:", f1_tuned)


## 48. Final Model Comparison Table

Compare the linear SVM, RBF SVM, and tuned SVM using the same test-set metrics.

In [ ]:
final_comparison = pd.DataFrame({
    "Model": ["Linear SVM", "RBF SVM", "Tuned SVM"],
    "Accuracy": [accuracy_linear, accuracy_rbf, accuracy_tuned],
    "Precision": [precision_linear, precision_rbf, precision_tuned],
    "Recall": [recall_linear, recall_rbf, recall_tuned],
    "F1 Score": [f1_linear, f1_rbf, f1_tuned]
})
final_comparison


## 49. Visualize Final Model Comparison

Plot the evaluation metrics for all three models. Interpret the chart together with per-class metrics and confusion matrices to judge whether tuning improves failure detection.

In [ ]:
final_comparison.set_index("Model").plot(kind="bar", figsize=(11, 6))
plt.title("Performance Comparison of Original and Tuned Models")
plt.xlabel("Model")
plt.ylabel("Score")
plt.ylim(0, 1.1)
plt.xticks(rotation=0)
plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.show()
